# 전처리 및 피처 선택

원본 데이터에 다음 단계를 적용합니다.

1. 농가 식별자(`frmDist`) 와 단일값 컬럼 제거
2. 컬럼명 정규화 (공백 → 언더스코어)
3. `date` 에서 시간 파생 피처(year/month/week/weekday) 생성
4. 수치형 피처에 `PowerTransformer` 스케일링 적용
5. SHAP 기반 피처 중요도와 상관계수를 활용해 타겟별 피처 셋(`X_yield`, `X_energy`) 분리
6. 결과를 후속 노트북에서 재사용할 수 있도록 `data/processed/` 에 저장


## Imports

In [ ]:
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import PowerTransformer
from lightgbm import LGBMRegressor
import shap

warnings.filterwarnings("ignore")
SEED = 42
np.random.seed(SEED)

## 데이터 로드 및 기본 정리

In [ ]:
DATA_DIR = Path("data")
PROCESSED_DIR = DATA_DIR / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

INPUT_CSV = DATA_DIR / "2023_smartFarm_AI_hackathon_dataset.csv"
TARGETS = ["outtrn_cumsum", "HeatingEnergyUsage_cumsum"]

raw = pd.read_csv(INPUT_CSV)
raw = raw.drop(columns=["frmDist"])
raw.columns = raw.columns.str.replace(" ", "_")

X = raw.drop(columns=TARGETS).copy()
Y = raw[TARGETS].copy()

# 단일값 컬럼 제거
multi_value_columns = X.columns[X.nunique() > 1]
X = X[multi_value_columns]

# date 가 있고 frmYear/frmWeek 는 date 와 정보 중복이라 제거
X = X.drop(columns=[c for c in ["frmYear", "frmWeek"] if c in X.columns])
print(f"X: {X.shape}, Y: {Y.shape}")

## 시간 파생 피처

In [ ]:
def add_calendar_features(frame: pd.DataFrame) -> pd.DataFrame:
    out = frame.copy()
    date = pd.to_datetime(out["date"], format="%Y%m%d")
    out["year"] = date.dt.year
    out["month"] = date.dt.month
    out["week"] = date.dt.isocalendar().week.astype(np.int32)
    out["weekday"] = date.dt.weekday
    out = out.drop(columns="date")
    return out


X_calendar = add_calendar_features(X)
X_calendar.head()

## 수치형 피처 스케일링

`PowerTransformer` 는 한쪽으로 치우친 분포를 가우시안에 가깝게 만들어 트리 기반 모델 외에도 선형 모델에 도움이 됩니다.


In [ ]:
scaler = PowerTransformer()
numeric_columns = X_calendar.select_dtypes(include=["number"]).columns.tolist()
X_scaled = X_calendar.copy()
X_scaled[numeric_columns] = scaler.fit_transform(X_scaled[numeric_columns])
X_scaled.describe().T.head()

## SHAP 기반 피처 중요도

타겟별로 LightGBM 을 한번씩 학습시키고 SHAP TreeExplainer 로 mean(|SHAP|) 을 구합니다.


In [ ]:
def shap_importance(features: pd.DataFrame, target: pd.Series, seed: int = SEED) -> pd.DataFrame:
    model = LGBMRegressor(random_state=seed, n_estimators=300, verbose=-1)
    model.fit(features, target)
    explainer = shap.TreeExplainer(model)
    shap_values = explainer.shap_values(features)
    mean_abs = np.abs(shap_values).mean(axis=0)
    return (
        pd.DataFrame({"feature": features.columns, "importance": mean_abs})
        .sort_values("importance", ascending=False)
        .reset_index(drop=True)
    )


importance_yield = shap_importance(X_scaled, Y["outtrn_cumsum"])
importance_energy = shap_importance(X_scaled, Y["HeatingEnergyUsage_cumsum"])

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
for ax, df, title in zip(axes, [importance_yield, importance_energy], TARGETS):
    top = df.head(15)
    ax.barh(top["feature"][::-1], top["importance"][::-1], color="steelblue")
    ax.set_title(f"{title} — top 15 SHAP importance")
plt.tight_layout()
plt.show()

## 상관계수 기반 보조 분석

SHAP 결과와 비교해 상관계수도 함께 확인합니다.


In [ ]:
correlations = pd.concat(
    {
        target: X_scaled.corrwith(Y[target]).abs().sort_values(ascending=False)
        for target in TARGETS
    },
    axis=1,
)
correlations.head(15)

## 타겟별 피처 셋 구성

SHAP 중요도 하위 컬럼을 제거해 타겟별로 슬림한 피처 셋을 만듭니다. 임계치는 SHAP 중요도 누적 비율 기준으로 결정합니다.


In [ ]:
def select_features(importance_df: pd.DataFrame, cumulative_threshold: float = 0.99) -> list[str]:
    sorted_df = importance_df.sort_values("importance", ascending=False).reset_index(drop=True)
    sorted_df["cumulative"] = sorted_df["importance"].cumsum() / sorted_df["importance"].sum()
    keep = sorted_df.loc[sorted_df["cumulative"] <= cumulative_threshold, "feature"].tolist()
    if not keep:
        keep = sorted_df["feature"].head(5).tolist()
    return keep


features_yield = select_features(importance_yield)
features_energy = select_features(importance_energy)

print(f"features_yield ({len(features_yield)}): {features_yield}")
print(f"features_energy ({len(features_energy)}): {features_energy}")

In [ ]:
X_yield = X_scaled[features_yield]
X_energy = X_scaled[features_energy]

X_yield.to_parquet(PROCESSED_DIR / "X_yield.parquet")
X_energy.to_parquet(PROCESSED_DIR / "X_energy.parquet")
Y.to_parquet(PROCESSED_DIR / "Y.parquet")
print("saved processed feature sets to", PROCESSED_DIR)

## 정리

- 단일값 컬럼·중복 컬럼·농가 식별자를 제거해 누설 방지와 입력 단순화를 했습니다.
- `PowerTransformer` 로 수치 피처 분포를 정규화했습니다.
- SHAP 중요도 누적 비율(99%) 기준으로 타겟별 피처 셋을 분리했습니다 (`X_yield`, `X_energy`).
- 후속 노트북은 저장된 parquet 을 그대로 불러와 모델 비교/튜닝에 사용합니다.
